# HR Analytics: Employee Attrition, Performance & Engagement Analysis

**Author:** Data Analytics Portfolio Project
**Dataset:** `Messy_HR_Dataset_Detailed.csv` (3,150 rows x 39 columns)

---

## 1. Project Overview

This project performs an end-to-end analysis of a company's Human Resources dataset. The raw data contains realistic
"messy" data quality issues — duplicate records, inconsistent date formats, whitespace errors in category labels, and
missing values — that are typical of real-world HR systems (HRIS exports, payroll systems, survey tools, and LMS
platforms merged together).

The notebook walks through the full analytics workflow:

1. **Data Inspection** — understand structure, quality, and completeness of the raw data
2. **Data Cleaning** — fix data quality issues with documented, justified decisions
3. **Exploratory Data Analysis (EDA)** — visualize workforce demographics, attrition, performance, engagement, and training
4. **Feature Engineering** — derive analytics-ready features (tenure, age groups, categories, flags)
5. **Business Insights** — translate findings into actionable HR recommendations
6. **Export** — save an analysis-ready cleaned dataset (`HR_Cleaned.csv`) for downstream BI tooling (e.g. Power BI)

## 2. Business Objective

HR and People Analytics teams need a clear, data-driven view of the workforce in order to answer questions such as:

- **Where is attrition concentrated**, and is it voluntary or involuntary?
- **Which departments, business units, or demographics** show performance or engagement risk?
- **Is the training program effective**, and where should the training budget be focused?
- **Are satisfaction, engagement, and work-life balance scores linked to attrition and performance?**

The goal of this notebook is to transform raw, messy HR data into a **clean, trustworthy dataset** and a set of
**actionable insights** that HR leadership can use to reduce attrition, improve engagement, and target training
investment more effectively.


## 3. Import Libraries

We use the standard Python data analytics stack: **Pandas** and **NumPy** for data manipulation, and **Matplotlib** /
**Seaborn** for visualization.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Visualization settings for consistent, portfolio-ready charts
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 11

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)

print("Libraries loaded successfully.")


## 4. Load Dataset

Loading the raw HR dataset. At this stage we keep everything as-is (no parsing/cleaning yet) so we can inspect the
data in its original, "messy" state first.


In [ ]:
df = pd.read_csv("Messy_HR_Dataset_Detailed.csv")
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


## 5. Data Inspection

Before cleaning anything, we need to understand the dataset: its shape, structure, data types, missing values,
duplicate records, and the distribution of key categorical fields. This step drives every decision made in the
Data Cleaning section.


In [ ]:
# Dataset shape
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")


In [ ]:
# Column names
df.columns.tolist()


In [ ]:
# Data types
df.dtypes


In [ ]:
# Missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_summary[missing_summary["Missing Count"] > 0].sort_values("Missing Count", ascending=False)


In [ ]:
# Duplicate rows
n_duplicates = df.duplicated().sum()
print(f"Fully duplicated rows: {n_duplicates}")

# Duplicate Employee IDs (should be unique identifiers)
n_dup_ids = df["Employee ID"].duplicated().sum()
print(f"Duplicate Employee IDs: {n_dup_ids}")


In [ ]:
# Descriptive statistics - numeric columns
df.describe()


In [ ]:
# Descriptive statistics - categorical/object columns
df.describe(include="object")


In [ ]:
# Unique values for key categorical columns
categorical_cols = [
    "EmployeeStatus", "EmployeeType", "PayZone", "EmployeeClassificationType",
    "TerminationType", "DepartmentType", "GenderCode", "RaceDesc", "MaritalDesc",
    "Performance Score", "BusinessUnit", "Training Type", "Training Outcome"
]

for col in categorical_cols:
    print(f"--- {col} ({df[col].nunique()} unique values) ---")
    print(df[col].unique())
    print()


### Inspection Findings

- The dataset has **3,150 rows and 39 columns** covering employee demographics, employment status, performance,
  engagement/satisfaction surveys, and training records.
- **`ExitDate`** and **`TerminationDescription`** are missing for ~1,544 rows — this is expected, since these fields
  are only populated for employees who have actually left the company (missingness = still employed, not a data
  quality problem).
- There are **150 fully duplicated rows** (identical across all 39 columns, including `Employee ID`) — these are
  true duplicate records and must be removed.
- **`DepartmentType`** contains inconsistent formatting (e.g., `"Production       "` with trailing whitespace).
- Date columns (`StartDate`, `ExitDate`, `DOB`, `Survey Date`, `Training Date`) are stored as **text strings**, not
  proper datetime objects, and use inconsistent day-month-year formats — these need to be parsed correctly.
- Categorical columns (`EmployeeStatus`, `PayZone`, `Performance Score`, etc.) are clean in terms of category labels,
  but should be validated for typos/whitespace during cleaning.


## 6. Data Cleaning

We now clean the dataset based on the issues identified during inspection. Every transformation below is
**explained and justified** — no data is removed or altered without a documented reason. We work on a copy
(`df_clean`) to preserve the original raw data.


In [ ]:
df_clean = df.copy()

# Drop the unnamed index column - it's just a leftover row index from the source export
# and duplicates the DataFrame's own index, so it carries no analytical value.
df_clean = df_clean.drop(columns=["Unnamed: 0"])
print("Dropped 'Unnamed: 0' (redundant export index column).")


In [ ]:
# --- Remove duplicate rows ---
# These 150 rows are fully identical across every column, including Employee ID,
# which should be a unique identifier. They are almost certainly duplicate exports
# from the source system rather than genuine repeat records, so we drop them.
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Removed {before - after} duplicate rows. New shape: {df_clean.shape}")


In [ ]:
# --- Standardize text/categorical columns ---
# Strip leading/trailing whitespace from all object (string) columns to fix issues
# like "Production       " vs "Production", which would otherwise be treated as
# separate categories.
str_cols = df_clean.select_dtypes(include=["object", "string"]).columns
for c in str_cols:
    df_clean[c] = df_clean[c].str.strip()

print("Stripped whitespace from all text columns.")
print(df_clean["DepartmentType"].unique())


In [ ]:
# --- Parse date columns correctly ---
# StartDate, ExitDate, Survey Date, Training Date use DD-Mon-YY / DD-MM-YYYY style formats.
# DOB uses DD-MM-YYYY. We convert all of these to proper datetime objects so they can be
# used for tenure/age calculations and time-based analysis.

date_formats = {
    "StartDate": "%d-%b-%y",
    "ExitDate": "%d-%b-%y",
    "Survey Date": "%d-%m-%Y",
    "Training Date": "%d-%b-%y",
    "DOB": "%d-%m-%Y",
}
date_cols_dayfirst = list(date_formats.keys())

for c, fmt in date_formats.items():
    df_clean[c] = pd.to_datetime(df_clean[c], format=fmt, errors="coerce")

df_clean[date_cols_dayfirst].dtypes


In [ ]:
# Check how many dates failed to parse (became NaT that weren't already missing)
for c in date_cols_dayfirst:
    original_missing = df[c].isnull().sum() if c in df.columns else 0
    new_missing = df_clean[c].isnull().sum()
    print(f"{c}: originally missing={original_missing}, missing after parsing={new_missing}")


**Note on `ExitDate`:** missing values are expected and meaningful (the employee is still active/has not left),
so we deliberately **do not impute or drop these rows** — a missing `ExitDate` is itself useful information and is
captured explicitly via the `Is Active` feature created later in Feature Engineering.


In [ ]:
# --- Fix / validate data types ---
# Ensure ID and score columns are proper numeric types (int), and rating/score
# columns are consistent. These generally already loaded correctly, but we
# make the intent explicit and guard against silent type issues.

int_cols = [
    "LocationCode", "Current Employee Rating", "Employee ID",
    "Engagement Score", "Satisfaction Score", "Work-Life Balance Score",
    "Training Duration(Days)"
]
for c in int_cols:
    df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce").astype("Int64")

df_clean["Training Cost"] = pd.to_numeric(df_clean["Training Cost"], errors="coerce")

print("Numeric columns validated and cast.")


In [ ]:
# --- Final missing value check after cleaning ---
missing_after = df_clean.isnull().sum()
missing_after[missing_after > 0]


In [ ]:
# --- Sanity check: confirm Employee ID is now unique ---
print(f"Employee ID duplicates remaining: {df_clean['Employee ID'].duplicated().sum()}")
print(f"Final cleaned shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")
df_clean.head()


### Cleaning Summary

| Issue | Action Taken | Justification |
|---|---|---|
| Redundant index column (`Unnamed: 0`) | Dropped | Duplicates the DataFrame index, no analytical value |
| 150 fully duplicate rows | Dropped | Identical across all columns including Employee ID — clear duplicate records |
| Whitespace in text columns (e.g. `DepartmentType`) | Stripped | Prevents false-distinct categories (`"Production"` vs `"Production   "`) |
| Date columns stored as text | Parsed to `datetime` | Required for tenure, age, and time-based analysis |
| Missing `ExitDate` / `TerminationDescription` | Kept as-is (not imputed) | Missingness is meaningful — indicates an active employee |
| Numeric columns loaded inconsistently | Cast to proper numeric/Int64 types | Ensures reliable aggregation and statistics |

No rows were removed for missingness — only true duplicate records were dropped. All other "missing" values are
legitimate and meaningful given the business context.


## 7. Exploratory Data Analysis (EDA)

With a clean dataset, we now explore the workforce across demographics, employment status, performance, engagement,
satisfaction, work-life balance, and training. Each chart is followed by a short set of business insights.


### 7.1 Employee Distribution by Department

In [ ]:
dept_counts = df_clean["DepartmentType"].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=dept_counts.values, y=dept_counts.index, hue=dept_counts.index,
            palette="viridis", legend=False)
plt.title("Employee Headcount by Department")
plt.xlabel("Number of Employees")
plt.ylabel("Department")
plt.tight_layout()
plt.show()

dept_counts


**Business Insights**
- Production is by far the largest department, accounting for the majority of total headcount — HR programs
  (training, engagement initiatives, retention efforts) here will have the largest organization-wide impact.
- IT/IS and Sales are the next largest functions, while Executive Office and Admin Offices are comparatively small,
  specialized teams.
- Because Production dominates headcount, department-level averages (e.g. average satisfaction) can be skewed by
  this single department — it's worth analyzing Production separately from smaller departments.


### 7.2 Gender Distribution

In [ ]:
gender_counts = df_clean["GenderCode"].value_counts()

plt.figure(figsize=(6, 6))
colors = sns.color_palette("Set2")
plt.pie(gender_counts.values, labels=gender_counts.index, autopct="%1.1f%%",
        colors=colors, startangle=90, wedgeprops={"edgecolor": "white"})
plt.title("Gender Distribution")
plt.tight_layout()
plt.show()

gender_counts


**Business Insights**
- The workforce gender split is close to balanced, which is a healthy starting point for diversity monitoring.
- This overall balance should be checked at the department and job-level (not shown here) to confirm there isn't
  gender concentration in specific roles or pay zones.


### 7.3 Attrition & Employment Status

In [ ]:
status_counts = df_clean["EmployeeStatus"].value_counts()

plt.figure(figsize=(9, 6))
sns.barplot(x=status_counts.values, y=status_counts.index, hue=status_counts.index,
            palette="magma", legend=False)
plt.title("Employee Status Breakdown")
plt.xlabel("Number of Employees")
plt.ylabel("Status")
plt.tight_layout()
plt.show()

status_counts


In [ ]:
# Attrition rate (terminated / total, excluding future-start employees who never started)
terminated_statuses = ["Voluntarily Terminated", "Terminated for Cause"]
attrition_count = df_clean["EmployeeStatus"].isin(terminated_statuses).sum()
attrition_rate = attrition_count / len(df_clean) * 100
print(f"Overall attrition rate: {attrition_rate:.1f}% ({attrition_count:,} of {len(df_clean):,} employees)")

# Voluntary vs involuntary termination type (for those who left)
term_type_counts = df_clean.loc[df_clean["TerminationType"] != "Unk", "TerminationType"].value_counts()
plt.figure(figsize=(8, 5))
sns.barplot(x=term_type_counts.index, y=term_type_counts.values, hue=term_type_counts.index,
            palette="rocket", legend=False)
plt.title("Termination Type (Employees Who Left)")
plt.xlabel("Termination Type")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


**Business Insights**
- The large majority of the workforce is Active, but **Voluntary Terminations are the dominant reason for attrition**
  — far more common than involuntary terminations or retirements.
- A high ratio of voluntary-to-involuntary terminations typically signals **retention issues** (compensation,
  engagement, management, career growth) rather than performance-driven exits, and warrants deeper investigation.
- "Leave of Absence" and "Future Start" employees should be tracked separately in workforce planning, as they are
  not part of the active, working headcount.


### 7.4 Performance Score Distribution

In [ ]:
perf_order = ["PIP", "Needs Improvement", "Fully Meets", "Exceeds"]
perf_counts = df_clean["Performance Score"].value_counts().reindex(perf_order)

plt.figure(figsize=(8, 5))
sns.barplot(x=perf_counts.index, y=perf_counts.values, hue=perf_counts.index,
            palette="crest", legend=False, order=perf_order)
plt.title("Performance Score Distribution")
plt.xlabel("Performance Score")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.show()

perf_counts


**Business Insights**
- The majority of employees fall into "Fully Meets" expectations, which is a healthy, expected distribution for a
  performing organization.
- A meaningful share of employees are in "Needs Improvement" or "PIP" (Performance Improvement Plan) — these
  employees represent both a **retention risk** and a **coaching opportunity** and should be cross-referenced with
  training completion and engagement scores.
- Relatively few employees are rated "Exceeds," suggesting there may be room to expand recognition or growth paths
  for top performers to keep them engaged and motivated.


### 7.5 Satisfaction Score Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x="Satisfaction Score", data=df_clean, hue="Satisfaction Score",
              palette="Blues", legend=False)
plt.title("Satisfaction Score Distribution (1 = Low, 5 = High)")
plt.xlabel("Satisfaction Score")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.show()

df_clean["Satisfaction Score"].describe()


**Business Insights**
- Satisfaction scores are fairly evenly spread across the 1–5 scale, indicating no single dominant sentiment —
  there isn't a clear organization-wide satisfaction crisis, but there is also no strong majority of highly
  satisfied employees.
- The presence of a sizable low-satisfaction group (scores of 1–2) is worth segmenting by department and manager
  to identify localized problem areas rather than treating satisfaction as a uniform, company-wide issue.


### 7.6 Engagement Score Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x="Engagement Score", data=df_clean, hue="Engagement Score",
              palette="Purples", legend=False)
plt.title("Engagement Score Distribution (1 = Low, 5 = High)")
plt.xlabel("Engagement Score")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.show()

df_clean["Engagement Score"].describe()


**Business Insights**
- Similar to satisfaction, engagement scores are broadly distributed, suggesting engagement varies significantly
  by individual/team rather than being a single company-wide trend.
- Employees with low engagement scores are a priority segment for HR intervention, since disengagement is a
  leading indicator of voluntary attrition and declining performance.


### 7.7 Work-Life Balance Score Distribution

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x="Work-Life Balance Score", data=df_clean, hue="Work-Life Balance Score",
              palette="Greens", legend=False)
plt.title("Work-Life Balance Score Distribution (1 = Low, 5 = High)")
plt.xlabel("Work-Life Balance Score")
plt.ylabel("Number of Employees")
plt.tight_layout()
plt.show()

df_clean["Work-Life Balance Score"].describe()


**Business Insights**
- Work-life balance scores show a wide spread, again without one score dominating — this metric should be
  monitored alongside engagement, since poor work-life balance is a common precursor to burnout and turnover.
- Departments with disproportionately low work-life balance scores (checked in cross-tab analysis) are good
  candidates for workload or scheduling review.


### 7.8 Training Cost Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_clean["Training Cost"], bins=30, kde=True, color="steelblue")
plt.title("Distribution of Training Cost per Session")
plt.xlabel("Training Cost ($)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

print(df_clean["Training Cost"].describe())


In [ ]:
# Average training cost and outcome by training type
train_summary = df_clean.groupby("Training Type", observed=True).agg(
    avg_cost=("Training Cost", "mean"),
    total_cost=("Training Cost", "sum"),
    sessions=("Training Cost", "count")
).round(2)
train_summary


In [ ]:
# Training outcome distribution
outcome_counts = df_clean["Training Outcome"].value_counts()

plt.figure(figsize=(8, 5))
sns.barplot(x=outcome_counts.index, y=outcome_counts.values, hue=outcome_counts.index,
            palette="cividis", legend=False)
plt.title("Training Outcome Distribution")
plt.xlabel("Outcome")
plt.ylabel("Number of Sessions")
plt.tight_layout()
plt.show()

outcome_counts


**Business Insights**
- Training costs are fairly widely distributed, with no extreme outliers dominating the budget — spend appears
  reasonably controlled across sessions.
- Training outcomes are split across Completed, Passed, Failed, and Incomplete — a non-trivial share of sessions
  end in **Failed** or **Incomplete**, representing wasted training spend that should be investigated (e.g. wrong
  training type for the role, poor delivery, scheduling conflicts).
- Comparing average cost against outcome quality can help identify whether **more expensive training programs
  actually produce better outcomes**, which is key for optimizing the training budget.


### 7.9 Correlation Analysis (Engagement, Satisfaction, Work-Life Balance, Performance)

In [ ]:
corr_cols = [
    "Engagement Score", "Satisfaction Score", "Work-Life Balance Score",
    "Current Employee Rating", "Training Duration(Days)", "Training Cost"
]
corr_matrix = df_clean[corr_cols].astype(float).corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f",
            linewidths=0.5)
plt.title("Correlation Matrix: Engagement, Satisfaction, Performance & Training Metrics")
plt.tight_layout()
plt.show()


**Business Insights**
- Correlations between engagement, satisfaction, work-life balance, and performance rating are weak in this
  dataset, suggesting these metrics behave largely independently of one another at the individual level (this is
  a realistic pattern in synthetic/anonymized HR data, and in practice would prompt HR to check for measurement
  or survey-timing issues if truly this weak).
- Training cost and training duration are only weakly related to performance rating, reinforcing the earlier point
  that training *spend* alone does not guarantee better outcomes — program design and relevance likely matter more
  than budget.


### 7.10 Attrition by Business Unit

In [ ]:
bu_status = pd.crosstab(df_clean["BusinessUnit"], df_clean["EmployeeStatus"])
bu_attrition_rate = (
    df_clean.assign(is_terminated=df_clean["EmployeeStatus"].isin(terminated_statuses))
    .groupby("BusinessUnit", observed=True)["is_terminated"]
    .mean()
    .mul(100)
    .round(1)
    .sort_values(ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=bu_attrition_rate.values, y=bu_attrition_rate.index, hue=bu_attrition_rate.index,
            palette="OrRd", legend=False)
plt.title("Attrition Rate (%) by Business Unit")
plt.xlabel("Attrition Rate (%)")
plt.ylabel("Business Unit")
plt.tight_layout()
plt.show()

bu_attrition_rate


**Business Insights**
- Attrition rate varies meaningfully across business units — the highest-attrition units should be prioritized for
  exit interviews and root-cause analysis (compensation benchmarking, manager effectiveness, workload).
- Business units with attrition well above the company average represent the highest-leverage opportunities for
  targeted retention programs, since fixing a concentrated problem is more cost-effective than a blanket
  company-wide initiative.


## 8. Feature Engineering

We now derive new, analysis-ready features that add business value: tenure, age, categorical bins for survey
scores, and an active-employee flag. These features make the dataset far more useful for downstream reporting
(e.g. in Power BI) and for slicing the EDA above by cohort.


In [ ]:
# --- Active Employee Flag ---
# True if the employee currently has Active status (as opposed to terminated, on leave, or future start)
df_clean["Is Active"] = df_clean["EmployeeStatus"] == "Active"


In [ ]:
# --- Employee Tenure (in years) ---
# For active employees: time from StartDate to today.
# For departed employees: time from StartDate to ExitDate.
# This gives an accurate "time with company" figure for every employee.
reference_date = pd.Timestamp(datetime.now().date())
end_date = df_clean["ExitDate"].fillna(reference_date)
df_clean["Tenure (Years)"] = ((end_date - df_clean["StartDate"]).dt.days / 365.25).round(2)

# Guard against any negative tenure from data issues
df_clean.loc[df_clean["Tenure (Years)"] < 0, "Tenure (Years)"] = np.nan

df_clean["Tenure (Years)"].describe()


In [ ]:
# --- Age & Age Group ---
df_clean["Age"] = ((reference_date - df_clean["DOB"]).dt.days / 365.25).round(0)

def age_group(age):
    if pd.isna(age):
        return "Unknown"
    elif age < 25:
        return "Under 25"
    elif age < 35:
        return "25-34"
    elif age < 45:
        return "35-44"
    elif age < 55:
        return "45-54"
    else:
        return "55+"

df_clean["Age Group"] = df_clean["Age"].apply(age_group)
df_clean["Age Group"].value_counts()


In [ ]:
# --- Satisfaction / Engagement / Work-Life Balance Categories ---
# Bucket the 1-5 survey scores into interpretable Low / Medium / High categories
# for easier business reporting and cross-tabulation.
def score_category(score):
    if pd.isna(score):
        return "Unknown"
    elif score <= 2:
        return "Low"
    elif score == 3:
        return "Medium"
    else:
        return "High"

df_clean["Satisfaction Category"] = df_clean["Satisfaction Score"].apply(score_category)
df_clean["Engagement Category"] = df_clean["Engagement Score"].apply(score_category)
df_clean["Work-Life Balance Category"] = df_clean["Work-Life Balance Score"].apply(score_category)

df_clean[["Satisfaction Category", "Engagement Category", "Work-Life Balance Category"]].apply(
    lambda col: col.value_counts()
)


In [ ]:
# --- Performance Category (simplified grouping for reporting) ---
performance_map = {
    "PIP": "At Risk",
    "Needs Improvement": "At Risk",
    "Fully Meets": "Solid Performer",
    "Exceeds": "Top Performer"
}
df_clean["Performance Category"] = df_clean["Performance Score"].map(performance_map)
df_clean["Performance Category"].value_counts()


In [ ]:
# Preview engineered features
feature_preview_cols = [
    "Employee ID", "Is Active", "Tenure (Years)", "Age", "Age Group",
    "Satisfaction Category", "Engagement Category", "Work-Life Balance Category",
    "Performance Category"
]
df_clean[feature_preview_cols].head(10)


### New Features Summary

| Feature | Description | Business Use |
|---|---|---|
| `Is Active` | Boolean flag for current active employment | Quickly filter active vs. departed workforce |
| `Tenure (Years)` | Years employed (Start to Exit/today) | Retention analysis, cohort comparisons |
| `Age` / `Age Group` | Employee age and 5-bucket age band | Demographic and generational workforce analysis |
| `Satisfaction / Engagement / Work-Life Balance Category` | Low / Medium / High bucket of 1-5 scores | Simplifies reporting and dashboard filtering |
| `Performance Category` | Groups performance scores into At Risk / Solid / Top Performer | Talent management and succession planning |


## 9. Business Insights

With engineered features in place, we can dig into a few higher-value, business-framed questions.


In [ ]:
# Attrition rate by tenure band - are we losing people early, or after many years?
tenure_bins = [0, 1, 3, 5, 10, 100]
tenure_labels = ["<1 yr", "1-3 yrs", "3-5 yrs", "5-10 yrs", "10+ yrs"]
df_clean["Tenure Band"] = pd.cut(df_clean["Tenure (Years)"], bins=tenure_bins, labels=tenure_labels)

tenure_attrition = (
    df_clean.assign(is_terminated=df_clean["EmployeeStatus"].isin(terminated_statuses))
    .groupby("Tenure Band", observed=True)["is_terminated"]
    .mean()
    .mul(100)
    .round(1)
)

plt.figure(figsize=(9, 5))
sns.barplot(x=tenure_attrition.index, y=tenure_attrition.values, hue=tenure_attrition.index,
            palette="flare", legend=False)
plt.title("Attrition Rate (%) by Tenure Band")
plt.xlabel("Tenure Band")
plt.ylabel("Attrition Rate (%)")
plt.tight_layout()
plt.show()

tenure_attrition


In [ ]:
# Average engagement/satisfaction for employees who left vs. stayed
engagement_by_status = df_clean.groupby("Is Active", observed=True)[
    ["Engagement Score", "Satisfaction Score", "Work-Life Balance Score"]
].mean().round(2)
engagement_by_status.index = ["Departed", "Active"]
engagement_by_status


In [ ]:
# Performance Category vs. Training Outcome - do at-risk performers also fail training more?
perf_training = pd.crosstab(df_clean["Performance Category"], df_clean["Training Outcome"], normalize="index").round(2) * 100
perf_training


### Key Business Insights

1. **Attrition is not concentrated in new hires alone** — reviewing attrition by tenure band highlights which
   career stage employees are most likely to leave, informing whether onboarding, mid-career growth, or long-tenure
   retention needs the most attention.
2. **Engagement, satisfaction, and work-life balance differ between active and departed employees** — even modest
   gaps here support the case that these survey metrics are meaningful early-warning indicators, and should be
   monitored continuously (e.g. via pulse surveys) rather than checked only during annual reviews.
3. **"At Risk" performers do not uniformly fail training** — cross-referencing performance category with training
   outcomes helps distinguish whether underperformance is a *skills gap* (fixable via training) or a *motivation/fit*
   issue (better addressed through coaching, role change, or performance management).
4. **Voluntary attrition outweighs involuntary attrition** company-wide, which is the single clearest signal that
   retention — not performance management — should be HR's top near-term priority.


## 10. Final Summary

**Data Quality:** The raw dataset required moderate cleaning — 150 duplicate rows were removed, text fields were
standardized, and 5 date columns were parsed from inconsistent string formats into proper datetime objects. No
records were dropped for missing data, since the only substantial missingness (`ExitDate`, `TerminationDescription`)
is legitimately tied to employees who have not left the company.

**Workforce Composition:** The organization is Production-heavy, with a roughly balanced gender split. The majority
of the workforce is currently Active, with attrition driven predominantly by **voluntary** turnover rather than
performance-based terminations.

**Performance & Engagement:** Most employees "Fully Meet" expectations, and survey metrics (engagement,
satisfaction, work-life balance) are broadly, evenly distributed — meaning risk is spread across the organization
rather than isolated to one obvious group, reinforcing the need for granular, department/manager-level monitoring
rather than only top-line company metrics.

**Training:** Training spend is distributed reasonably across sessions, but a meaningful share of sessions end in
Failed or Incomplete outcomes — an opportunity to improve ROI on the training budget.

**Recommended Next Steps for HR:**
- Investigate high-attrition business units and tenure bands with targeted exit interviews.
- Treat engagement/satisfaction/work-life-balance scores as leading indicators and monitor them continuously, not
  just annually.
- Audit low-outcome training programs to identify whether the issue is program design, delivery, or targeting.
- Use the `Performance Category` and `Tenure Band` features to build a simple, ongoing retention-risk watchlist.

## 11. Export Cleaned Dataset

The cleaned, feature-engineered dataset is exported as `HR_Cleaned.csv` for use in downstream BI tools (e.g. Power BI)
and future analysis.


In [ ]:
output_path = "HR_Cleaned.csv"
df_clean.to_csv(output_path, index=False)
print(f"Cleaned dataset exported to '{output_path}'")
print(f"Final shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")


## 12. Recommended Power BI Dashboard Ideas (Not Implemented Here)

This notebook intentionally stops at producing a clean, analysis-ready dataset. The following dashboard concepts
are natural next steps for a Power BI (or Tableau) layer built on top of `HR_Cleaned.csv`:

1. **Workforce Overview Dashboard** — headcount by department/business unit, gender & age-group mix, active vs.
   departed KPI cards.
2. **Attrition & Retention Dashboard** — attrition rate trend over time, by department/business unit/tenure band,
   voluntary vs. involuntary breakdown, with drill-through to individual exit reasons.
3. **Performance & Engagement Dashboard** — performance category distribution, engagement/satisfaction/work-life
   balance heatmaps by department and manager, correlation with attrition.
4. **Training ROI Dashboard** — training spend by program/type, completion & pass rates, cost vs. outcome analysis,
   and training impact on performance rating over time.
5. **Diversity & Inclusion Dashboard** — gender, race, and marital-status composition by department, pay zone, and
   job level, with year-over-year trend tracking.

*(These are proposed for a future project phase and are not built in this notebook.)*
